# Assignment: Improving Generalization in Deep Learning — Solution

**Course:** Machine Learning with Big Data
**Format:** Teams of 2–3
**Dataset:** CIFAR-100
**Deliverables:** Reproducible code, experiment results, and a technical report

> **Instructor solution.** Run the three setup cells first. After that, every part is standalone: its cells contain their own model, optimizer and full training loop, and depend only on the setup cells, so any part can be run on its own or copied into a separate notebook. Within a part, the "repeat with 3 seeds" cell uses the best setting found by that part's sweep cell.
>
> The model answers explain the mechanism and the pattern to expect; the exact numbers come from your run. Set `QUICK_RUN = True` to check everything end to end in a few minutes before the full run.

## Objective

Build an image classifier and investigate how different training strategies affect generalization. You will study regularization, early stopping, noise, label smoothing, data augmentation, multi-task learning, transfer learning, ensembling, dropout, and weight noise.

Your goal is to **demonstrate a reproducible improvement over a baseline and explain which methods contributed to it**. Individual methods may fail to improve performance; careful negative results are valid.

## Dataset and prediction tasks

CIFAR-100 contains 60,000 color images, each 32 × 32 pixels, organized into 100 fine-grained classes and 20 broader categories.

| Split | Size | Source | Use |
|---|---|---|---|
| Train | 40,000 (400 per class) | Official training set, stratified, `seed=42` | Fitting models and normalization statistics |
| Validation | 10,000 (100 per class) | Remaining official training images | All tuning, model selection and early stopping |
| Test | 10,000 | Official test set | Evaluated **once**, in Part 11 |

All experiments use the same split, saved to `splits.npz`.

The primary task is **100-class classification**. For multi-task learning, the model also predicts each image's **20-class coarse category**.

Dataset and category information: [CIFAR-100](https://www.cs.toronto.edu/~kriz/cifar.html).

## Setup (run these three cells first)

In [ ]:
# !pip install datasets
import os
import time
import copy
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
import matplotlib.pyplot as plt
from datasets import load_dataset

plt.style.use("dark_background")

QUICK_RUN = False                       # True: 3 epochs, one seed, a quarter of the training data

EPOCHS = 3 if QUICK_RUN else 25         # the same training budget for every MLP experiment
SEEDS = [0] if QUICK_RUN else [0, 1, 2]

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

In [ ]:
ds = load_dataset("uoft-cs/cifar100")
fine_names = ds["train"].features["fine_label"].names
coarse_names = ds["train"].features["coarse_label"].names

X_all = np.stack([np.asarray(im.convert("RGB")) for im in ds["train"]["img"]])      # (50000, 32, 32, 3) uint8
yf_all = np.array(ds["train"]["fine_label"])
yc_all = np.array(ds["train"]["coarse_label"])
X_test = np.stack([np.asarray(im.convert("RGB")) for im in ds["test"]["img"]])      # not used until Part 11
yf_test = np.array(ds["test"]["fine_label"])

# One stratified split: 20% of every class goes to validation. Saved so every run uses the same split.
if os.path.exists("splits.npz"):
    saved = np.load("splits.npz")
    train_idx, val_idx = saved["train_idx"], saved["val_idx"]
else:
    rng = np.random.default_rng(42)
    val_idx = []
    for c in range(100):
        idx_c = rng.permutation(np.flatnonzero(yf_all == c))
        val_idx.extend(idx_c[: round(0.2 * len(idx_c))])
    val_idx = np.sort(np.array(val_idx))
    train_idx = np.setdiff1d(np.arange(len(yf_all)), val_idx)
    np.savez("splits.npz", train_idx=train_idx, val_idx=val_idx)

if QUICK_RUN:   # keep a quarter of every class
    rng = np.random.default_rng(0)
    train_idx = np.sort(np.concatenate([rng.permutation(train_idx[yf_all[train_idx] == c])[: max(1, (yf_all[train_idx] == c).sum() // 4)]
                                        for c in range(100)]))

# Normalization statistics from the training split only
train_float = torch.tensor(X_all[train_idx], dtype=torch.float32) / 255
MEAN = train_float.mean(dim=(0, 1, 2))
STD = train_float.std(dim=(0, 1, 2))

Xtr = ((train_float - MEAN) / STD).permute(0, 3, 1, 2).contiguous()          # (N, 3, 32, 32)
Xval = ((torch.tensor(X_all[val_idx], dtype=torch.float32) / 255 - MEAN) / STD).permute(0, 3, 1, 2).contiguous()
Ytr, Ctr = torch.tensor(yf_all[train_idx]), torch.tensor(yc_all[train_idx])
Yval, Cval = torch.tensor(yf_all[val_idx]), torch.tensor(yc_all[val_idx])
del train_float

val_loader = DataLoader(TensorDataset(Xval, Yval), batch_size=1000)

print(f"train {len(Ytr)} | validation {len(Yval)} | test {len(yf_test)}")
print("mean", MEAN.numpy().round(4), "std", STD.numpy().round(4))

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(14, 11))
for c, ax in enumerate(axes.flat):
    i = train_idx[np.flatnonzero(yc_all[train_idx] == c)[0]]
    ax.imshow(X_all[i])
    ax.set_title(f"$y_i$ = {yf_all[i]}\nFine: {fine_names[yf_all[i]]}\nCoarse: {coarse_names[c]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

**Fixes compared with the earlier draft:** one stratified split used everywhere (the draft made two different splits and dropped the coarse labels); seeds set for every run; normalization statistics from the training split; and every experiment uses the baseline's learning rate (1e-3) and batch size (128) unless the part says otherwise. The draft's L2 run used `lr=1e-4`, so its improvement could not be separated from the learning-rate change.

Throughout, *training* accuracy is the accuracy on the training batches during each epoch, as in the original baseline. With dropout, noise or augmentation this is measured on perturbed inputs, so for those methods the train − validation gap slightly understates how well the model fits clean training images.

## Part 0: Establish a baseline

A sequential feed-forward network (3072 → 512 → 256 → 100, ReLU) trained from random initialization with cross-entropy loss.

- Normalized inputs; no augmentation, dropout, label smoothing, injected noise, or weight decay.
- Adam, learning rate 1e-3, batch size 128, `EPOCHS` epochs, final weights kept.
- Training and validation loss and accuracy recorded after every epoch.

**Generalization gap = training accuracy − validation accuracy.**

In [ ]:
rows, histories = [], {}
for seed in SEEDS:
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    histories[seed] = history

    rows.append({"seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
baseline_df = pd.DataFrame(rows)
display(baseline_df.round(4))
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"baseline validation accuracy {100 * baseline_df.val_acc.mean():.2f} ± {100 * baseline_df.val_acc.std():.2f} %, "
      f"gap {100 * baseline_df.gap.mean():.1f} pts")

history = histories[SEEDS[0]]
epochs = range(1, len(history["val_acc"]) + 1)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].fill_between(epochs, history["train_loss"], history["val_loss"], alpha=0.3, hatch="//", label="Loss generalization gap")
axs[0].plot(epochs, history["train_loss"], label="Train loss")
axs[0].plot(epochs, history["val_loss"], label="Validation loss")
axs[0].set_xlabel("Epoch"); axs[0].set_ylabel("Loss"); axs[0].legend()
axs[1].fill_between(epochs, history["train_acc"], history["val_acc"], alpha=0.3, hatch="//", label="Accuracy generalization gap")
axs[1].plot(epochs, history["train_acc"], label="Train accuracy")
axs[1].plot(epochs, history["val_acc"], label="Validation accuracy")
axs[1].set_xlabel("Epoch"); axs[1].set_ylabel("Accuracy"); axs[1].legend()
plt.tight_layout()
plt.show()
print(f"lowest validation loss at epoch {int(np.argmin(history['val_loss'])) + 1}, "
      f"highest validation accuracy at epoch {int(np.argmax(history['val_acc'])) + 1}")

**Discussion (model answer).** The baseline overfits strongly. Training accuracy keeps climbing for the whole budget (above 60% by epoch 25 in the draft run), while validation accuracy stalls near 20% after about 8–10 epochs. Validation loss reaches its minimum within the first few epochs and then rises steeply, even though validation accuracy barely changes. The network keeps memorizing training images and becomes increasingly confident on the validation images it gets wrong; cross-entropy punishes confident mistakes, accuracy does not. An MLP on raw pixels has no built-in notion that a shifted or mirrored image is the same object, so it has to learn each training image almost separately, which is why the gap opens so quickly. The baseline is not deliberately weakened; it is a reasonable MLP with standard settings.

## Part 1: Weight-penalty regularization

Add an L2 penalty to the loss, then compare it with decoupled weight decay.

$$
L = L_{CE} + \lambda \sum_i w_i^2
$$

- Sweep λ ∈ {1e-5, 1e-4, 5e-4, 1e-3, 5e-3}, implemented as an explicit penalty with Adam. λ = 0 is included as the baseline.
- Repeat the best λ using `AdamW(weight_decay=...)` and compare. Explain why L2-in-the-loss and decoupled decay are not equivalent under Adam.
- Optional: L1 penalty; report the fraction of near-zero weights (|w| < 1e-3).

The penalty covers the weight matrices only, not the biases.

In [ ]:
rows = []
for lam in [0, 1e-5, 1e-4, 5e-4, 1e-3, 5e-3]:
    seed = 0
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            l2_penalty = sum(p.pow(2).sum() for p in model.parameters() if p.dim() > 1)
            loss = criterion(outputs, targets) + lam * l2_penalty
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"lambda": lam, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
sweep_df = pd.DataFrame(rows)
display(sweep_df.round(4))
best_l2 = sweep_df.loc[sweep_df.val_acc.idxmax(), "lambda"]
print("best λ:", best_l2)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
labels = [str(v) for v in sweep_df["lambda"]]
axs[0].plot(labels, 100 * sweep_df.val_acc, "o-", label="validation")
axs[0].plot(labels, 100 * sweep_df.train_acc, "s--", label="train")
axs[0].set_ylabel("accuracy (%)")
axs[0].legend()
axs[1].plot(labels, sweep_df.val_loss, "o-")
axs[1].set_ylabel("validation loss")
axs[2].plot(labels, 100 * sweep_df.gap, "o-")
axs[2].set_ylabel("train − validation accuracy (pts)")
for ax in axs:
    ax.set_xlabel("lambda")
fig.suptitle("Explicit L2 penalty with Adam")
plt.tight_layout()
plt.show()

In [ ]:
# Decoupled weight decay. With plain SGD, λ·Σw² in the loss equals decay of 2λ per unit learning rate,
# so AdamW(weight_decay=2λ) is the 'same' setting on paper. Larger values are what AdamW is usually run with.
rows = []
for wd in [2 * best_l2, 1e-2, 5e-2, 1e-1]:
    seed = 0
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=wd)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"weight_decay": wd, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
sweep_df = pd.DataFrame(rows)
display(sweep_df.round(4))
best_wd = sweep_df.loc[sweep_df.val_acc.idxmax(), "weight_decay"]
print("best weight_decay:", best_wd)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
labels = [str(v) for v in sweep_df["weight_decay"]]
axs[0].plot(labels, 100 * sweep_df.val_acc, "o-", label="validation")
axs[0].plot(labels, 100 * sweep_df.train_acc, "s--", label="train")
axs[0].set_ylabel("accuracy (%)")
axs[0].legend()
axs[1].plot(labels, sweep_df.val_loss, "o-")
axs[1].set_ylabel("validation loss")
axs[2].plot(labels, 100 * sweep_df.gap, "o-")
axs[2].set_ylabel("train − validation accuracy (pts)")
for ax in axs:
    ax.set_xlabel("weight_decay")
fig.suptitle("Decoupled weight decay (AdamW)")
plt.tight_layout()
plt.show()

In [ ]:
# Repeat the best explicit-L2 setting and the best AdamW setting with every seed
rows = []
for seed in SEEDS:
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            l2_penalty = sum(p.pow(2).sum() for p in model.parameters() if p.dim() > 1)
            loss = criterion(outputs, targets) + best_l2 * l2_penalty
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
label = f"L2 in the loss, λ={best_l2}"

seed_df = pd.DataFrame(rows)
display(seed_df.round(4))
print(f"{label}: validation accuracy {100 * seed_df.val_acc.mean():.2f} ± {100 * seed_df.val_acc.std():.2f} %, "
      f"validation loss {seed_df.val_loss.mean():.3f} ± {seed_df.val_loss.std():.3f}, "
      f"gap {100 * seed_df.gap.mean():.1f} pts")

rows = []
for seed in SEEDS:
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=best_wd)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
label = f"AdamW, weight_decay={best_wd}"

seed_df = pd.DataFrame(rows)
display(seed_df.round(4))
print(f"{label}: validation accuracy {100 * seed_df.val_acc.mean():.2f} ± {100 * seed_df.val_acc.std():.2f} %, "
      f"validation loss {seed_df.val_loss.mean():.3f} ± {seed_df.val_loss.std():.3f}, "
      f"gap {100 * seed_df.gap.mean():.1f} pts")

In [ ]:
# Optional: L1 penalty and sparsity (λ = 0 is the baseline, for comparison)
rows = []
for lam in [0, 1e-5]:
    seed = 0
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            l1_penalty = sum(p.abs().sum() for p in model.parameters() if p.dim() > 1)
            loss = criterion(outputs, targets) + lam * l1_penalty
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    weights = torch.cat([p.detach().flatten() for p in model.parameters() if p.dim() > 1])
    rows.append({"l1_lambda": lam, "val_acc": history["val_acc"][-1],
                 "fraction |w| < 1e-3": (weights.abs() < 1e-3).float().mean().item()})
display(pd.DataFrame(rows).round(4))

**Discussion (model answer).**

*Effect.* A well-chosen L2 penalty mostly fixes the loss curve rather than the accuracy: validation loss stops climbing after its minimum, the gap shrinks, and validation accuracy improves by a modest amount. Too small a λ does nothing; too large a λ (5e-3) underfits, and training and validation accuracy fall together. Discouraging large weights does reduce overfitting, because the baseline's growing confidence comes from growing logit scale, which is exactly what the penalty limits.

*Why L2-in-the-loss and AdamW differ.* With an L2 term the decay enters as part of the gradient, 2λw. Adam then divides every gradient coordinate by the running root-mean-square of that coordinate's gradients. Weights that have seen large data gradients receive almost no decay, and weights with tiny gradients receive a lot, so the regularization strength gets tangled with the optimizer's per-parameter scaling. AdamW applies the decay outside the adaptive step, w ← w − η·wd·w, shrinking every weight by the same fraction regardless of its gradient history (Loshchilov & Hutter, 2019). In the sweep, the "on-paper equivalent" wd = 2λ is far too weak under AdamW, and AdamW needs values around 1e-2 to 1e-1. Matching λ values across the two methods is not meaningful; each must be tuned on its own.

*L1.* L1 drives many weights to (near) exactly zero, so the fraction of |w| < 1e-3 rises sharply compared with the baseline. Sparsity is a different goal from accuracy: L1 usually helps accuracy less than L2 here.

## Part 2: Early stopping

Monitor validation loss each epoch and stop when it hasn't improved for `patience` epochs, restoring the best weights.

- Sweep patience ∈ {3, 5, 10, 20}. "None" is the baseline (full budget, final weights).
- Report the stopping epoch, the restored epoch, and the compute saved relative to the full `EPOCHS` budget.
- Discuss: does stopping on validation *loss* vs. validation *accuracy* pick different epochs? Why?

In [ ]:
rows, histories = [], {}
for patience in [None, 3, 5, 10, 20]:
    seed = 0
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_loss, best_epoch, wait = float("inf"), 0, 0
    best_state = copy.deepcopy(model.state_dict())
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
        if history["val_loss"][-1] < best_loss:
            best_loss, best_epoch, wait = history["val_loss"][-1], epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if patience is not None and wait >= patience:
                break
    train_time = time.time() - start
    stopped_epoch = len(history["val_acc"])
    if patience is not None:
        model.load_state_dict(best_state)          # restore the best checkpoint
        final = best_epoch - 1                     # report the metrics of the restored epoch
    else:
        final = stopped_epoch - 1

    histories[patience] = history
    rows.append({"patience": str(patience), "stopped at epoch": stopped_epoch, "restored epoch": best_epoch if patience else stopped_epoch,
                 "compute saved (%)": round(100 * (1 - stopped_epoch / EPOCHS), 1),
                 "train_acc": history["train_acc"][final], "val_acc": history["val_acc"][final],
                 "val_loss": history["val_loss"][final], "gap": history["train_acc"][final] - history["val_acc"][final]})
sweep_df = pd.DataFrame(rows)
display(sweep_df.round(4))
candidates = sweep_df[sweep_df.patience != "None"]
best_patience = int(candidates.loc[candidates.val_acc.idxmax(), "patience"])
print("best patience:", best_patience)

h = histories[None]
print(f"full run: lowest validation loss at epoch {int(np.argmin(h['val_loss'])) + 1}, "
      f"highest validation accuracy at epoch {int(np.argmax(h['val_acc'])) + 1}")

In [ ]:
# Same best patience, but monitoring validation accuracy instead of validation loss
rows = []
for monitor in ["val_loss", "val_acc"]:
    seed = 0
    patience = best_patience
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_score, best_epoch, wait = float("inf"), 0, 0
    best_state = copy.deepcopy(model.state_dict())
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
        score = history["val_loss"][-1] if monitor == "val_loss" else -history["val_acc"][-1]
        if score < best_score:
            best_score, best_epoch, wait = score, epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= patience:
                break
    train_time = time.time() - start
    model.load_state_dict(best_state)
    rows.append({"monitor": monitor, "restored epoch": best_epoch, "stopped at epoch": len(history["val_acc"]),
                 "val_acc": history["val_acc"][best_epoch - 1], "val_loss": history["val_loss"][best_epoch - 1]})
display(pd.DataFrame(rows).round(4))

In [ ]:
# Repeat the best patience with every seed
rows = []
for seed in SEEDS:
    patience = best_patience
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_loss, best_epoch, wait = float("inf"), 0, 0
    best_state = copy.deepcopy(model.state_dict())
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
        if history["val_loss"][-1] < best_loss:
            best_loss, best_epoch, wait = history["val_loss"][-1], epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if patience is not None and wait >= patience:
                break
    train_time = time.time() - start
    stopped_epoch = len(history["val_acc"])
    if patience is not None:
        model.load_state_dict(best_state)          # restore the best checkpoint
        final = best_epoch - 1                     # report the metrics of the restored epoch
    else:
        final = stopped_epoch - 1

    rows.append({"seed": seed, "train_acc": history["train_acc"][final], "val_acc": history["val_acc"][final],
                 "val_loss": history["val_loss"][final], "gap": history["train_acc"][final] - history["val_acc"][final],
                 "stopped at epoch": stopped_epoch, "train_time_s": round(train_time, 1)})
label = f"early stopping, patience={best_patience}"

seed_df = pd.DataFrame(rows)
display(seed_df.round(4))
print(f"{label}: validation accuracy {100 * seed_df.val_acc.mean():.2f} ± {100 * seed_df.val_acc.std():.2f} %, "
      f"validation loss {seed_df.val_loss.mean():.3f} ± {seed_df.val_loss.std():.3f}, "
      f"gap {100 * seed_df.gap.mean():.1f} pts")

**Discussion (model answer).** Early stopping on validation loss stops very early for this model, because validation loss bottoms out within the first few epochs. With patience 3–5 it saves most of the training budget and returns a model whose validation loss is far better than the final baseline weights, with accuracy similar to or slightly below the baseline's peak. Larger patience lets training run past the minimum, saves less compute, and still restores the same early checkpoint.

The two criteria pick different epochs. Accuracy peaks *later* than loss: after the loss minimum, the network keeps fitting training images, which still turns a few more validation predictions from wrong to right, so accuracy creeps up. At the same time the logits grow, so the validation images it still gets wrong are assigned very low probability, and the loss rises. Stopping on accuracy gives slightly higher accuracy and noticeably worse loss and calibration; stopping on loss gives the better-calibrated model.

One caution: the validation set chooses the stopping epoch and also reports the result, so these validation numbers are slightly optimistic. The test evaluation in Part 11 is the unbiased check.

## Part 3: Noise on inputs

During training only, add Gaussian noise to each normalized input batch: `x + σ·randn_like(x)`.

- Sweep σ ∈ {0.01, 0.05, 0.1, 0.2, 0.4}; σ = 0 is the baseline.
- Connect your result to the known equivalence between small input noise and a Tikhonov (L2-type) penalty on the input gradient.

Noise is added **after** normalization, so σ is in units of one channel standard deviation (σ = 0.1 is about 7/255 in pixel units). Validation images are never noised.

In [ ]:
rows = []
for sigma in [0, 0.01, 0.05, 0.1, 0.2, 0.4]:
    seed = 0
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = inputs + sigma * torch.randn_like(inputs)       # training inputs only
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"sigma": sigma, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
sweep_df = pd.DataFrame(rows)
display(sweep_df.round(4))
best_sigma = sweep_df.loc[sweep_df.val_acc.idxmax(), "sigma"]
print("best σ:", best_sigma)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
labels = [str(v) for v in sweep_df["sigma"]]
axs[0].plot(labels, 100 * sweep_df.val_acc, "o-", label="validation")
axs[0].plot(labels, 100 * sweep_df.train_acc, "s--", label="train")
axs[0].set_ylabel("accuracy (%)")
axs[0].legend()
axs[1].plot(labels, sweep_df.val_loss, "o-")
axs[1].set_ylabel("validation loss")
axs[2].plot(labels, 100 * sweep_df.gap, "o-")
axs[2].set_ylabel("train − validation accuracy (pts)")
for ax in axs:
    ax.set_xlabel("sigma")
fig.suptitle("Gaussian noise on training inputs")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for seed in SEEDS:
    sigma = best_sigma
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = inputs + sigma * torch.randn_like(inputs)       # training inputs only
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
label = f"input noise, σ={best_sigma}"

seed_df = pd.DataFrame(rows)
display(seed_df.round(4))
print(f"{label}: validation accuracy {100 * seed_df.val_acc.mean():.2f} ± {100 * seed_df.val_acc.std():.2f} %, "
      f"validation loss {seed_df.val_loss.mean():.3f} ± {seed_df.val_loss.std():.3f}, "
      f"gap {100 * seed_df.gap.mean():.1f} pts")

**Discussion (model answer).** For small σ, the expected loss on noisy inputs expands as

$$\mathbb{E}_\epsilon\,[L(x+\epsilon)] \approx L(x) + \tfrac{\sigma^2}{2}\,\mathrm{tr}\big(\nabla_x^2 L\big),$$

and for squared error this reduces to a penalty on $\|\nabla_x f(x)\|^2$ (Bishop, 1995): training with noise is approximately Tikhonov regularization that asks the output to change slowly as the input changes. Very small σ (0.01) is almost no regularization and matches the baseline within seed noise. Moderate σ gives a small improvement in the gap and validation loss. Large σ (0.4) masks real image detail, and both training and validation accuracy drop. Input noise is a weak regularizer here because random pixel noise is not the kind of variation that separates CIFAR training and validation images (those differ in object pose, position and background), which is why augmentation in Part 5 helps much more.

## Part 4: Label smoothing

Replace one-hot targets with a mixture of the one-hot vector and the uniform distribution over K = 100 classes.

$$
y_k^{LS} = (1 - \varepsilon)\, y_k + \frac{\varepsilon}{K}
$$

- Sweep ε ∈ {0.05, 0.1, 0.2, 0.3}; ε = 0 is the baseline. The loss is written by hand, then checked against `nn.CrossEntropyLoss(label_smoothing=ε)`.
- Report expected calibration error (ECE) with 15 bins and draw a reliability diagram for the baseline and best ε.
- Validation loss is always ordinary cross-entropy against the hard labels, so it is comparable with the baseline.

In [ ]:
# Check the hand-written smoothed loss against PyTorch's built-in version
logits, y = torch.randn(64, 100), torch.randint(0, 100, (64,))
for eps in [0.0, 0.1, 0.3]:
    log_probs = F.log_softmax(logits, dim=1)
    ours = ((1 - eps) * -log_probs.gather(1, y[:, None]).squeeze(1) + eps * -log_probs.mean(dim=1)).mean()
    ref = nn.CrossEntropyLoss(label_smoothing=eps)(logits, y)
    print(f"ε={eps}: ours {ours:.6f} | PyTorch {ref:.6f}")
    assert torch.allclose(ours, ref, atol=1e-6)

In [ ]:
rows, val_probs = [], {}
for eps in [0, 0.05, 0.1, 0.2, 0.3]:
    seed = 0
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            log_probs = F.log_softmax(outputs, dim=1)
            loss = ((1 - eps) * -log_probs.gather(1, targets[:, None]).squeeze(1) + eps * -log_probs.mean(dim=1)).mean()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    # Expected calibration error on the validation set, 15 equal-width confidence bins
    with torch.no_grad():
        probs = torch.cat([F.softmax(model(x.to(device)), dim=1).cpu() for x, _ in val_loader])
    val_probs[eps] = probs
    confidence, prediction = probs.max(dim=1)
    is_correct = (prediction == Yval).float()
    ece = 0.0
    bin_edges = torch.linspace(0, 1, 16)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (confidence > lo) & (confidence <= hi)
        if in_bin.any():
            ece += in_bin.float().mean().item() * abs(confidence[in_bin].mean().item() - is_correct[in_bin].mean().item())

    rows.append({"epsilon": eps, "ece": ece, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
sweep_df = pd.DataFrame(rows)
display(sweep_df.round(4))
best_eps = sweep_df.loc[sweep_df.val_acc.idxmax(), "epsilon"]
print("best ε (by validation accuracy):", best_eps)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
labels = [str(v) for v in sweep_df["epsilon"]]
axs[0].plot(labels, 100 * sweep_df.val_acc, "o-", label="validation")
axs[0].plot(labels, 100 * sweep_df.train_acc, "s--", label="train")
axs[0].set_ylabel("accuracy (%)")
axs[0].legend()
axs[1].plot(labels, sweep_df.val_loss, "o-")
axs[1].set_ylabel("validation loss")
axs[2].plot(labels, 100 * sweep_df.gap, "o-")
axs[2].set_ylabel("train − validation accuracy (pts)")
for ax in axs:
    ax.set_xlabel("epsilon")
fig.suptitle("Label smoothing")
plt.tight_layout()
plt.show()

# Reliability diagram: accuracy against confidence in each bin
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, eps_list in [(axs[0], [0, best_eps]), (axs[1], list(val_probs))]:
    ax.plot([0, 1], [0, 1], ":", color="gray")
    for eps in eps_list:
        confidence, prediction = val_probs[eps].max(dim=1)
        is_correct = (prediction == Yval).float()
        xs, ys = [], []
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            in_bin = (confidence > lo) & (confidence <= hi)
            if in_bin.any():
                xs.append(confidence[in_bin].mean().item())
                ys.append(is_correct[in_bin].mean().item())
        ece = sweep_df.loc[sweep_df.epsilon == eps, "ece"].item()
        ax.plot(xs, ys, "o-", label=f"ε={eps} (ECE {ece:.3f})")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.legend()
axs[0].set_title("Baseline vs best ε"); axs[1].set_title("All ε")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for seed in SEEDS:
    eps = best_eps
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(label_smoothing=eps)     # verified identical to the hand-written loss above
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
label = f"label smoothing, ε={best_eps}"

seed_df = pd.DataFrame(rows)
display(seed_df.round(4))
print(f"{label}: validation accuracy {100 * seed_df.val_acc.mean():.2f} ± {100 * seed_df.val_acc.std():.2f} %, "
      f"validation loss {seed_df.val_loss.mean():.3f} ± {seed_df.val_loss.std():.3f}, "
      f"gap {100 * seed_df.gap.mean():.1f} pts")

**Discussion (model answer).** Label smoothing caps how confident the network is rewarded for being: the optimal logit gap between the true class and the others becomes finite. On this overfit baseline, validation loss falls a lot compared with the final baseline weights, and accuracy usually improves a little, because the model is no longer pushed to extreme logits to memorize training images.

The reliability diagram shows the calibration effect. The baseline curve lies well *below* the diagonal (overconfident: at 90% confidence it is right far less than 90% of the time). Small ε pulls the curve toward the diagonal and cuts ECE sharply. Large ε (0.3) overshoots: the curve moves *above* the diagonal (underconfident) and ECE rises again. The ε that minimizes ECE is often smaller than the ε that maximizes accuracy, so the report should say which criterion chose ε. Smoothed *training* loss is not comparable with the baseline's training loss; only hard-label loss on held-out data is.

## Part 5: Data augmentation

Apply augmentations to training images only.

- (a) Random crop (32×32 with 4-pixel padding) + horizontal flip.
- (b) (a) + Cutout (an 8×8 square erased at a random position), implemented with `RandomErasing`.
- Ablation: crop only, flip only, both. "none" is the baseline.
- Explain which invariances each augmentation encodes and why vertical flips would be a poor choice here.

The transforms run on the normalized tensors, so the crop padding and the Cutout square are filled with 0, which is the average colour.

In [ ]:
from torchvision import transforms


class AugmentedDataset(Dataset):
    """Applies `transform` to each training image every time it is loaded."""

    def __init__(self, X, y, transform):
        self.X, self.y, self.transform = X, y, transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.transform(self.X[i]), self.y[i]


cutout = transforms.RandomErasing(p=1.0, scale=(64 / 1024, 64 / 1024), ratio=(1.0, 1.0), value=0)
augmentations = {
    "none":             transforms.Compose([]),
    "crop":             transforms.RandomCrop(32, padding=4),
    "flip":             transforms.RandomHorizontalFlip(),
    "crop+flip":        transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()]),
    "crop+flip+cutout": transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), cutout]),
}

rows = []
for aug_name in augmentations:
    seed = 0
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(AugmentedDataset(Xtr, Ytr, augmentations[aug_name]), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"augmentation": aug_name, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
sweep_df = pd.DataFrame(rows)
display(sweep_df.round(4))
best_aug = sweep_df.loc[sweep_df.val_acc.idxmax(), "augmentation"]
print("best augmentation:", best_aug)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].bar(sweep_df.augmentation, 100 * sweep_df.val_acc)
axs[0].set_ylabel("validation accuracy (%)")
axs[1].bar(sweep_df.augmentation, 100 * sweep_df.gap)
axs[1].set_ylabel("train − validation accuracy (pts)")
for ax in axs:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

# What the augmentations look like (un-normalized for display)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
img = Xtr[0]
for ax, (name, t) in zip(axes, [("original", transforms.Compose([]))] + list(augmentations.items())[1:]):
    shown = (t(img).permute(1, 2, 0) * STD + MEAN).clamp(0, 1)
    ax.imshow(shown.numpy()); ax.set_title(name, fontsize=9); ax.axis("off")
plt.show()

In [ ]:
from torchvision import transforms


class AugmentedDataset(Dataset):
    """Applies `transform` to each training image every time it is loaded."""

    def __init__(self, X, y, transform):
        self.X, self.y, self.transform = X, y, transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.transform(self.X[i]), self.y[i]


cutout = transforms.RandomErasing(p=1.0, scale=(64 / 1024, 64 / 1024), ratio=(1.0, 1.0), value=0)
augmentations = {
    "none":             transforms.Compose([]),
    "crop":             transforms.RandomCrop(32, padding=4),
    "flip":             transforms.RandomHorizontalFlip(),
    "crop+flip":        transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()]),
    "crop+flip+cutout": transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), cutout]),
}

rows = []
for seed in SEEDS:
    aug_name = best_aug
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(AugmentedDataset(Xtr, Ytr, augmentations[aug_name]), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
label = f"augmentation: {best_aug}"

seed_df = pd.DataFrame(rows)
display(seed_df.round(4))
print(f"{label}: validation accuracy {100 * seed_df.val_acc.mean():.2f} ± {100 * seed_df.val_acc.std():.2f} %, "
      f"validation loss {seed_df.val_loss.mean():.3f} ± {seed_df.val_loss.std():.3f}, "
      f"gap {100 * seed_df.gap.mean():.1f} pts")

**Discussion (model answer).**

*Invariances.* Random crops with padding encode *small translation invariance*: an object shifted by a few pixels is the same object. Horizontal flips encode *left–right mirror symmetry*: a mirrored dog is still a dog. Cutout encodes *robustness to occlusion*: the label should survive when part of the object is hidden.

*Results.* Augmentation is typically the most effective single method for this model. An MLP on pixels has no convolution or pooling, so it cannot get translation invariance from its architecture; crops supply it through the data and give the larger share of the gain in the ablation, with flips adding a smaller, independent amount. Training accuracy rises much more slowly and the gap is much smaller, because the network never sees exactly the same image twice. Within a fixed 25-epoch budget, Cutout may underfit and finish no better than crop+flip; stronger augmentation usually needs longer training to pay off.

*Vertical flips.* Natural photographs have a canonical "up" (sky above, ground below). A vertically flipped image is not a plausible CIFAR image, so training on it teaches an invariance the test data never needs and spends capacity on unrealistic inputs. An augmentation is only useful if the transformed image could plausibly appear in the data with the same label.

## Part 6: Multi-task learning

Share the network trunk and attach two heads: a fine head (100 classes) and a coarse head (20 superclasses).

$$
L = L_{fine} + \alpha\, L_{coarse}
$$

- Sweep α ∈ {0.1, 0.25, 0.5, 1.0, 2.0}; α = 0 is the baseline (the coarse head exists but is not trained).
- Model selection uses **fine** validation accuracy only.
- Report fine and coarse accuracy, and how often the fine prediction's superclass disagrees with the coarse prediction.

In [ ]:
class TwoHeadMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        self.fine_head = nn.Linear(256, 100)
        self.coarse_head = nn.Linear(256, 20)

    def forward(self, x):
        z = self.trunk(x)
        return self.fine_head(z), self.coarse_head(z)


fine_to_coarse = torch.zeros(100, dtype=torch.long)       # superclass of every fine class
fine_to_coarse[Ytr] = Ctr
mt_val_loader = DataLoader(TensorDataset(Xval, Yval, Cval), batch_size=1000)

rows = []
for alpha in [0, 0.1, 0.25, 0.5, 1.0, 2.0]:
    seed = 0
    torch.manual_seed(seed)
    model = TwoHeadMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr, Ctr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets, coarse_targets in train_loader:
            inputs, targets, coarse_targets = inputs.to(device), targets.to(device), coarse_targets.to(device)
            optimizer.zero_grad()
            fine_out, coarse_out = model(inputs)
            loss = criterion(fine_out, targets) + alpha * criterion(coarse_out, coarse_targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (fine_out.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, coarse_correct, disagree, count = 0.0, 0, 0, 0, 0
        with torch.no_grad():
            for inputs, targets, coarse_targets in mt_val_loader:
                inputs, targets, coarse_targets = inputs.to(device), targets.to(device), coarse_targets.to(device)
                fine_out, coarse_out = model(inputs)
                loss_sum += F.cross_entropy(fine_out, targets).item() * len(targets)
                correct += (fine_out.argmax(1) == targets).sum().item()
                coarse_correct += (coarse_out.argmax(1) == coarse_targets).sum().item()
                disagree += (fine_to_coarse.to(device)[fine_out.argmax(1)] != coarse_out.argmax(1)).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"alpha": alpha, "coarse_acc": coarse_correct / count, "disagreement": disagree / count, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})

sweep_df = pd.DataFrame(rows)
display(sweep_df.round(4))
candidates = sweep_df[sweep_df.alpha > 0]
best_alpha = candidates.loc[candidates.val_acc.idxmax(), "alpha"]
print("best α:", best_alpha)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
labels = [str(v) for v in sweep_df["alpha"]]
axs[0].plot(labels, 100 * sweep_df.val_acc, "o-", label="validation")
axs[0].plot(labels, 100 * sweep_df.train_acc, "s--", label="train")
axs[0].set_ylabel("accuracy (%)")
axs[0].legend()
axs[1].plot(labels, sweep_df.val_loss, "o-")
axs[1].set_ylabel("validation loss")
axs[2].plot(labels, 100 * sweep_df.gap, "o-")
axs[2].set_ylabel("train − validation accuracy (pts)")
for ax in axs:
    ax.set_xlabel("alpha")
fig.suptitle("Multi-task: fine + α · coarse")
plt.tight_layout()
plt.show()

In [ ]:
class TwoHeadMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        self.fine_head = nn.Linear(256, 100)
        self.coarse_head = nn.Linear(256, 20)

    def forward(self, x):
        z = self.trunk(x)
        return self.fine_head(z), self.coarse_head(z)


fine_to_coarse = torch.zeros(100, dtype=torch.long)       # superclass of every fine class
fine_to_coarse[Ytr] = Ctr
mt_val_loader = DataLoader(TensorDataset(Xval, Yval, Cval), batch_size=1000)

rows = []
for seed in SEEDS:
    alpha = best_alpha
    torch.manual_seed(seed)
    model = TwoHeadMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr, Ctr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets, coarse_targets in train_loader:
            inputs, targets, coarse_targets = inputs.to(device), targets.to(device), coarse_targets.to(device)
            optimizer.zero_grad()
            fine_out, coarse_out = model(inputs)
            loss = criterion(fine_out, targets) + alpha * criterion(coarse_out, coarse_targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (fine_out.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, coarse_correct, disagree, count = 0.0, 0, 0, 0, 0
        with torch.no_grad():
            for inputs, targets, coarse_targets in mt_val_loader:
                inputs, targets, coarse_targets = inputs.to(device), targets.to(device), coarse_targets.to(device)
                fine_out, coarse_out = model(inputs)
                loss_sum += F.cross_entropy(fine_out, targets).item() * len(targets)
                correct += (fine_out.argmax(1) == targets).sum().item()
                coarse_correct += (coarse_out.argmax(1) == coarse_targets).sum().item()
                disagree += (fine_to_coarse.to(device)[fine_out.argmax(1)] != coarse_out.argmax(1)).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"coarse_acc": coarse_correct / count, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})

label = f"multi-task, α={best_alpha}"

seed_df = pd.DataFrame(rows)
display(seed_df.round(4))
print(f"{label}: validation accuracy {100 * seed_df.val_acc.mean():.2f} ± {100 * seed_df.val_acc.std():.2f} %, "
      f"validation loss {seed_df.val_loss.mean():.3f} ± {seed_df.val_loss.std():.3f}, "
      f"gap {100 * seed_df.gap.mean():.1f} pts")

**Discussion (model answer).** The coarse labels add no new information about any individual image, because each coarse label is a fixed function of the fine label. What they add is *structure*: they tell the network which fine classes belong together (maple, oak, palm, pine and willow are all trees). The coarse loss pushes the shared representation to place those classes near each other, which acts as a regularizer (an extra constraint on the trunk that discourages memorizing idiosyncratic features) and as extra supervision about similarity between classes; it behaves somewhat like structured label smoothing.

The effect on fine accuracy is usually small: positive or neutral for small to moderate α, and negative for large α (2.0), where the trunk starts optimizing for the easier 20-way task at the expense of fine distinctions. The disagreement rate between the two heads falls as α grows, meaning the heads become more consistent. With α = 0 the coarse head is untrained, so its accuracy is near chance. Frame this in the report as additional *structured* supervision, not an independent source of data, so large gains should not be expected.

## Part 7: Transfer learning

ResNet-18 pretrained on ImageNet (`torchvision.models.resnet18(weights="IMAGENET1K_V1")`), with the final layer replaced by a 100-way classifier. Inputs are upsampled from 32×32 to `TL_SIZE`×`TL_SIZE` and converted to ImageNet normalization inside the model.

- (a) Linear probe: freeze the backbone, train only the new head (BatchNorm statistics also frozen).
- (b) Full fine-tuning with a smaller backbone learning rate (1e-4 backbone, 1e-3 head).
- (c) Same ResNet-18 architecture trained from scratch, same input size, as a control for the architecture change.
- Budget: (a) and (b) get `TL_EPOCHS` (about a third of the baseline budget); (c) gets the full `EPOCHS`. One seed each because of cost.

This is the most expensive part: expect a few minutes per epoch on a Colab T4 at 128×128.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms

model = models.resnet18(
    weights=models.ResNet18_Weights.IMAGENET1K_V1
)
model.fc = nn.Linear(model.fc.in_features, 100)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/fsultan/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:01<00:00, 39.6MB/s]


In [4]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Note that the last layer of the model is `model.fc`, which is a `nn.Linear` layer with `in_features` equal to the output features of the previous layer and `out_features` equal to 100.

We want to replace the last layer with a new `nn.Linear` layer that has the same `in_features` but `out_features` equal to the number of classes in our dataset, which is 100.

```python
model.fc = nn.Linear(model.fc.in_features, 100)
```



In [ ]:
from torchvision.models import resnet18

TL_SIZE = 64 if QUICK_RUN else 128
TL_EPOCHS = max(1, EPOCHS // 3)
PRETRAINED_WEIGHTS = "IMAGENET1K_V1"
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])


class ResNetForCIFAR(nn.Module):
    def __init__(self, pretrained, freeze_backbone):
        super().__init__()
        self.net = resnet18(weights=PRETRAINED_WEIGHTS if pretrained else None)
        self.net.fc = nn.Linear(self.net.fc.in_features, 100)
        self.freeze_backbone = freeze_backbone
        if freeze_backbone:
            for name, p in self.net.named_parameters():
                p.requires_grad = name.startswith("fc.")
        # converts CIFAR-normalized inputs to ImageNet-normalized inputs
        self.register_buffer("scale", (STD / IMAGENET_STD).view(1, 3, 1, 1))
        self.register_buffer("shift", ((MEAN - IMAGENET_MEAN) / IMAGENET_STD).view(1, 3, 1, 1))

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_backbone:      # a frozen backbone must also keep its BatchNorm statistics fixed
            self.net.eval()
            self.net.fc.train(mode)
        return self

    def forward(self, x):
        x = x * self.scale + self.shift
        x = F.interpolate(x, size=TL_SIZE, mode="bilinear", align_corners=False)
        return self.net(x)


tl_val_loader = DataLoader(TensorDataset(Xval, Yval), batch_size=250)
setups = {
    "(a) linear probe": dict(pretrained=True, freeze=True, epochs=TL_EPOCHS),
    "(b) fine-tune": dict(pretrained=True, freeze=False, epochs=TL_EPOCHS),
    "(c) from scratch": dict(pretrained=False, freeze=False, epochs=EPOCHS),
}
rows, histories = [], {}
for setup_name, setup in setups.items():
    seed = 0
    torch.manual_seed(seed)
    model = ResNetForCIFAR(setup["pretrained"], setup["freeze"]).to(device)
    if setup_name == "(b) fine-tune":
        backbone = [p for n, p in model.net.named_parameters() if not n.startswith("fc.")]
        optimizer = torch.optim.Adam([{"params": backbone, "lr": 1e-4},
                                      {"params": model.net.fc.parameters(), "lr": 1e-3}])
    else:
        optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(setup["epochs"]):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in tl_val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
        print(f"{setup_name} epoch {epoch + 1}: train acc {history['train_acc'][-1]:.2%}, val acc {history['val_acc'][-1]:.2%}")
    train_time = time.time() - start
    histories[setup_name] = history
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    rows.append({"setup": setup_name, "epochs": setup["epochs"], "trainable params": trainable, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "wall-clock (min)": round(train_time / 60, 1)})
display(pd.DataFrame(rows).round(4))

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for setup_name, h in histories.items():
    axs[0].plot(range(1, len(h["val_acc"]) + 1), 100 * np.array(h["val_acc"]), "o-", label=f"{setup_name} val")
    axs[0].plot(range(1, len(h["train_acc"]) + 1), 100 * np.array(h["train_acc"]), "--", label=f"{setup_name} train")
    axs[1].plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], "o-", label=setup_name)
axs[0].set_ylabel("accuracy (%)"); axs[1].set_ylabel("validation loss")
for ax in axs:
    ax.set_xlabel("epoch"); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Discussion (model answer).**

*What the runs show.* Fine-tuning (b) is by far the strongest model in the assignment, despite using about a third of the epochs. The linear probe (a) beats the MLP baseline but falls well short of (b). The from-scratch ResNet (c) beats the MLP (convolutions and pooling build in translation invariance) but stays far below (b) and overfits within its budget. Comparing (b) with (c), not with the MLP, is what separates the value of *pretraining* from the value of a better *architecture*.

*Why (b) beats (c) with fewer epochs.* The pretrained weights were learned from about 1.28 million ImageNet images. The early layers already detect edges, colours and textures, and later layers detect object parts, all useful for CIFAR. Fine-tuning starts close to a good solution and only has to adapt it, so it converges in a few epochs, and the pretrained features act as a strong prior that limits overfitting on 40,000 images.

*Why the linear probe is limited.* Upsampled 32×32 images are blurry and statistically different from ImageNet photos, so the frozen features are not ideally suited; fine-tuning lets the backbone adapt. Freezing BatchNorm statistics matters: updating them while the weights are frozen silently changes the features.

*When transfer can fail.* When the target domain is far from the source (spectrograms, medical scans, multispectral satellite data), pretrained features may not help and can even hurt (negative transfer). Too high a fine-tuning learning rate can destroy the pretrained features in the first epochs. The cost is also much higher: ResNet-18 has about 11 million parameters and needs far more computation per image than the MLP.

## Part 8: Ensembling

Train 5 baseline networks that differ only in random seed, and average their softmax outputs.

- Plot validation accuracy and loss against ensemble size 1–5.
- Repeat with 5 copies of the best single regularized configuration. Set `DROPOUT_P` to the best setting from Part 9, or replace the model with your best configuration from Parts 1–10.
- Compare the ensemble with its average member and its best member, and report training and inference cost.

In [ ]:
DROPOUT_P = 0.25          # best single regularized configuration; change to your result from Part 9
N_MEMBERS = 3 if QUICK_RUN else 5

member_probs = {"baseline": [], f"dropout p={DROPOUT_P}": []}
train_times = []
for config in member_probs:
    p = 0.0 if config == "baseline" else DROPOUT_P
    for seed in range(N_MEMBERS):
        torch.manual_seed(seed)
        model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, 512),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(256, 100),
        ).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                                  generator=torch.Generator().manual_seed(seed))
        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
        start = time.time()
        for epoch in range(EPOCHS):
            model.train()
            loss_sum, correct, count = 0.0, 0, 0
            for inputs, targets in train_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()
                loss_sum += loss.item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
            history["train_loss"].append(loss_sum / count)
            history["train_acc"].append(correct / count)

            model.eval()
            loss_sum, correct, count = 0.0, 0, 0
            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs, targets = inputs.to(device), targets.to(device)
                    outputs = model(inputs)
                    loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                    correct += (outputs.argmax(1) == targets).sum().item()
                    count += len(targets)
            history["val_loss"].append(loss_sum / count)
            history["val_acc"].append(correct / count)
        train_time = time.time() - start
        train_times.append(train_time)
        with torch.no_grad():
            member_probs[config].append(torch.cat([F.softmax(model(x.to(device)), dim=1).cpu() for x, _ in val_loader]))

rows = []
for config, probs in member_probs.items():
    for k in range(1, len(probs) + 1):
        avg = torch.stack(probs[:k]).mean(dim=0)
        rows.append({"config": config, "ensemble size": k,
                     "val_acc": (avg.argmax(1) == Yval).float().mean().item(),
                     "val_loss": -torch.log(avg[torch.arange(len(Yval)), Yval].clamp_min(1e-12)).mean().item()})
ens_df = pd.DataFrame(rows)
display(ens_df.round(4))

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for config, g in ens_df.groupby("config"):
    axs[0].plot(g["ensemble size"], 100 * g.val_acc, "o-", label=config)
    axs[1].plot(g["ensemble size"], g.val_loss, "o-", label=config)
axs[0].set_ylabel("validation accuracy (%)"); axs[1].set_ylabel("validation loss")
for ax in axs:
    ax.set_xlabel("ensemble size"); ax.legend()
plt.tight_layout()
plt.show()

for config, probs in member_probs.items():
    member_accs = [(p.argmax(1) == Yval).float().mean().item() for p in probs]
    ens_acc = (torch.stack(probs).mean(0).argmax(1) == Yval).float().mean().item()
    print(f"{config}: average member {np.mean(member_accs):.2%}, best member {np.max(member_accs):.2%}, "
          f"ensemble of {len(probs)} {ens_acc:.2%}")

start = time.time()
with torch.no_grad():
    for x, _ in val_loader:
        model(x.to(device))
inference_time = time.time() - start
print(f"one model: {np.mean(train_times):.0f} s to train, {1000 * inference_time:.0f} ms to predict 10,000 images. "
      f"An ensemble of {N_MEMBERS} costs {N_MEMBERS}× both, and stores {N_MEMBERS}× the parameters.")

**Discussion (model answer).** Averaging probabilities from independently trained networks reduces the variance part of the error: each network overfits in a different, partly random way, and those errors partly cancel. Accuracy rises with ensemble size with diminishing returns (most of the gain comes from the first 2–3 members), and the ensemble beats both its average member and its best member. Validation loss improves much more than accuracy, because averaging pulls overconfident wrong predictions back toward moderate probabilities, so ensembles also improve calibration. The gain is larger for the overfit baseline than for the regularized configuration, because there is more variance to remove, but the regularized ensemble remains the better model.

The cost is linear in the number of members: K times the training time, inference time and storage. That can be worth it for offline scoring and usually is not for latency-sensitive inference. Snapshot ensembling (saving checkpoints at the low points of a cyclic learning rate) gives part of the benefit for the training cost of one run, but inference still costs K forward passes.

## Part 9: Dropout

Add dropout before each fully connected layer after the first (that is, after each hidden ReLU).

- Sweep p ∈ {0.1, 0.25, 0.5, 0.7}; p = 0 is the baseline.
- Variant: dropout on the input pixels, p ∈ {0.1, 0.2}. (Channel dropout, `Dropout2d`, is for convolutional networks and does not apply to this MLP.)
- Show what happens if you forget `model.eval()` at evaluation time.
- Optional: Monte Carlo dropout (20 stochastic passes); does it improve validation loss or ECE?

In [ ]:
rows, models = [], {}
for p_input, p in [(0, 0), (0, 0.1), (0, 0.25), (0, 0.5), (0, 0.7), (0.1, 0), (0.2, 0)]:
    seed = 0
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Dropout(p_input),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Dropout(p),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(p),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    models[(p_input, p)] = model

    rows.append({"input dropout": p_input, "hidden dropout": p, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
sweep_df = pd.DataFrame(rows)
display(sweep_df.round(4))
hidden = sweep_df[sweep_df["input dropout"] == 0].copy()
best_p = hidden.loc[hidden.val_acc.idxmax(), "hidden dropout"]
print("best hidden dropout p:", best_p)
sweep_df = hidden

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
labels = [str(v) for v in sweep_df["hidden dropout"]]
axs[0].plot(labels, 100 * sweep_df.val_acc, "o-", label="validation")
axs[0].plot(labels, 100 * sweep_df.train_acc, "s--", label="train")
axs[0].set_ylabel("accuracy (%)")
axs[0].legend()
axs[1].plot(labels, sweep_df.val_loss, "o-")
axs[1].set_ylabel("validation loss")
axs[2].plot(labels, 100 * sweep_df.gap, "o-")
axs[2].set_ylabel("train − validation accuracy (pts)")
for ax in axs:
    ax.set_xlabel("hidden dropout")
fig.suptitle("Dropout on hidden layers")
plt.tight_layout()
plt.show()

In [ ]:
# Forgetting model.eval(), and Monte Carlo dropout, on the best dropout model from the sweep
model = models[(0, best_p)]


def val_metrics(probs):
    correct = (probs.argmax(1) == Yval).float()
    confidence = probs.max(1).values
    ece = 0.0
    bin_edges = torch.linspace(0, 1, 16)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (confidence > lo) & (confidence <= hi)
        if in_bin.any():
            ece += in_bin.float().mean().item() * abs(confidence[in_bin].mean().item() - correct[in_bin].mean().item())
    loss = -torch.log(probs[torch.arange(len(Yval)), Yval].clamp_min(1e-12)).mean().item()
    return {"val_acc": correct.mean().item(), "val_loss": loss, "ECE": ece}


with torch.no_grad():
    model.eval()
    probs_eval = torch.cat([F.softmax(model(x.to(device)), 1).cpu() for x, _ in val_loader])
    model.train()                       # the mistake: dropout still active at prediction time
    probs_forgot = torch.cat([F.softmax(model(x.to(device)), 1).cpu() for x, _ in val_loader])
    probs_mc = torch.stack([torch.cat([F.softmax(model(x.to(device)), 1).cpu() for x, _ in val_loader])
                            for _ in range(20)]).mean(0)
    model.eval()
display(pd.DataFrame({"model.eval()": val_metrics(probs_eval), "forgot model.eval()": val_metrics(probs_forgot),
                      "MC dropout, 20 passes": val_metrics(probs_mc)}).round(4))

In [ ]:
rows = []
for seed in SEEDS:
    p_input, p = 0, best_p
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Dropout(p_input),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Dropout(p),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(p),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
label = f"dropout, p={best_p}"

seed_df = pd.DataFrame(rows)
display(seed_df.round(4))
print(f"{label}: validation accuracy {100 * seed_df.val_acc.mean():.2f} ± {100 * seed_df.val_acc.std():.2f} %, "
      f"validation loss {seed_df.val_loss.mean():.3f} ± {seed_df.val_loss.std():.3f}, "
      f"gap {100 * seed_df.gap.mean():.1f} pts")

**Discussion (model answer).** Dropout randomly removes hidden units during training, so no unit can rely on specific partners being present. That breaks up co-adapted, memorizing feature combinations, and training becomes an implicit average over many thinned networks. Moderate dropout (0.25–0.5) reduces the gap clearly and improves validation accuracy and loss; p = 0.7 removes so much capacity that the network underfits within the 25-epoch budget. Input dropout acts more like pixel noise and helps less.

Forgetting `model.eval()` leaves dropout active at prediction time. Each prediction then comes from one random thinned network instead of the full network (PyTorch rescales kept units by 1/(1−p) during training so that the full network matches the average), so accuracy drops and results change from run to run. Averaging 20 such passes (MC dropout) recovers the accuracy of the deterministic model and often gives slightly better validation loss and ECE, at 20 times the inference cost. In networks with BatchNorm, MC dropout must switch on only the dropout layers, not BatchNorm's training mode.

## Part 10: Noise on weights

At every training step, perturb the weights, compute the gradient at the perturbed point, restore the original weights, then take the optimizer step.

$$
\tilde{w} = w + \epsilon, \quad \epsilon \sim \mathcal{N}\left(0, (\sigma\, |w|)^2\right)
$$

- Noise is relative to each weight's magnitude. Sweep σ ∈ {0.001, 0.01, 0.05, 0.1}; σ = 0 is the baseline.
- Noise is applied to weight matrices only, not biases or BatchNorm parameters. Fresh noise is drawn every step, and the clean weights are restored before the update, so noise never accumulates.
- Validation always uses the clean weights.

In [ ]:
rows = []
for sigma in [0, 0.001, 0.01, 0.05, 0.1]:
    seed = 0
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    weight_matrices = [p for p in model.parameters() if p.dim() > 1]      # not biases (or BatchNorm parameters)
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            clean_weights = [w.detach().clone() for w in weight_matrices]
            with torch.no_grad():
                for w in weight_matrices:
                    w.add_(sigma * w.abs() * torch.randn_like(w))
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            with torch.no_grad():                  # gradient was computed at the noisy weights; update the clean ones
                for w, clean in zip(weight_matrices, clean_weights):
                    w.copy_(clean)
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"sigma": sigma, "seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
sweep_df = pd.DataFrame(rows)
display(sweep_df.round(4))
best_wn = sweep_df.loc[sweep_df.val_acc.idxmax(), "sigma"]
print("best σ:", best_wn)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
labels = [str(v) for v in sweep_df["sigma"]]
axs[0].plot(labels, 100 * sweep_df.val_acc, "o-", label="validation")
axs[0].plot(labels, 100 * sweep_df.train_acc, "s--", label="train")
axs[0].set_ylabel("accuracy (%)")
axs[0].legend()
axs[1].plot(labels, sweep_df.val_loss, "o-")
axs[1].set_ylabel("validation loss")
axs[2].plot(labels, 100 * sweep_df.gap, "o-")
axs[2].set_ylabel("train − validation accuracy (pts)")
for ax in axs:
    ax.set_xlabel("sigma")
fig.suptitle("Multiplicative Gaussian noise on weights")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for seed in SEEDS:
    sigma = best_wn
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    weight_matrices = [p for p in model.parameters() if p.dim() > 1]      # not biases (or BatchNorm parameters)
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=128, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            clean_weights = [w.detach().clone() for w in weight_matrices]
            with torch.no_grad():
                for w in weight_matrices:
                    w.add_(sigma * w.abs() * torch.randn_like(w))
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            with torch.no_grad():                  # gradient was computed at the noisy weights; update the clean ones
                for w, clean in zip(weight_matrices, clean_weights):
                    w.copy_(clean)
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)
    train_time = time.time() - start
    rows.append({"seed": seed, "train_acc": history["train_acc"][-1], "val_acc": history["val_acc"][-1],
                 "val_loss": history["val_loss"][-1], "gap": history["train_acc"][-1] - history["val_acc"][-1],
                 "train_time_s": round(train_time, 1)})
label = f"weight noise, σ={best_wn}"

seed_df = pd.DataFrame(rows)
display(seed_df.round(4))
print(f"{label}: validation accuracy {100 * seed_df.val_acc.mean():.2f} ± {100 * seed_df.val_acc.std():.2f} %, "
      f"validation loss {seed_df.val_loss.mean():.3f} ± {seed_df.val_loss.std():.3f}, "
      f"gap {100 * seed_df.gap.mean():.1f} pts")

**Discussion (model answer).**

*Flat minima.* Averaged over the noise, the training objective becomes $\mathbb{E}_\epsilon[L(w+\epsilon)] \approx L(w) + \tfrac12\sum_i \sigma^2 w_i^2\, \partial^2 L/\partial w_i^2$. The extra term penalizes curvature of the loss along each weight, scaled by the weight's size, so the optimizer is rewarded for solutions where the loss stays low in a neighbourhood of w. Such flat minima tend to generalize better (Hochreiter & Schmidhuber, 1997); sharpness-aware minimization (SAM) is the worst-case version of the same idea.

*Results.* σ = 0.001 is too small to matter. Moderate σ gives a small reduction in the gap; the effect is usually smaller than dropout's, whose noise is much larger (it zeroes whole units). Large σ makes the gradients noisy enough to slow learning and lowers accuracy.

*Why not BatchNorm.* BatchNorm's scale γ and shift β multiply or offset an entire channel, so noise on them perturbs every unit in the channel at once, a much larger effect than independent noise on individual connections, and it destabilizes training. BatchNorm's running statistics are not trained parameters and must stay clean. This MLP has no BatchNorm, but the code follows the same rule (only parameters with two or more dimensions are perturbed).

## Part 11: Best combined model, ablation, and final test evaluation

1. Combine at least four techniques at their best validation settings. Fill in `COMBINED` from your Parts 1–10 results; the values below are placeholders. Transfer learning and ensembling are left out so the combined model stays an MLP (needed for the fair comparison in Part 12); ensembling can be applied on top of any model. Multi-task learning is left out to keep the training loop simple; add it if it helped in Part 6.
2. **Leave-one-out ablation:** switch off each technique in turn and retrain with all seeds.
3. Identify redundant or interfering techniques.
4. Freeze every choice, then evaluate on the **test set once**: accuracy, loss, ECE, a reliability diagram, and robustness to Gaussian input noise at two severities chosen in advance.

In [ ]:
from torchvision import transforms


class AugmentedDataset(Dataset):
    def __init__(self, X, y, transform):
        self.X, self.y, self.transform = X, y, transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.transform(self.X[i]), self.y[i]


crop_and_flip = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()])


def train_config(config, seed, epochs=EPOCHS, train_positions=None):
    """Trains the MLP with whichever techniques are switched on in `config`.
    Returns the trained model, its history, and the train/validation numbers of the returned weights."""
    X, Y = Xtr, Ytr
    if train_positions is not None:
        X, Y = Xtr[train_positions], Ytr[train_positions]
    torch.manual_seed(seed)
    p = config["dropout"]
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Dropout(p),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(p),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=config["weight_decay"])
    criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])
    dataset = AugmentedDataset(X, Y, crop_and_flip) if config["augment"] else TensorDataset(X, Y)
    train_loader = DataLoader(dataset, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(seed))
    weight_matrices = [w for w in model.parameters() if w.dim() > 1]
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_loss, best_epoch, wait = float("inf"), 0, 0
    best_state = copy.deepcopy(model.state_dict())
    start = time.time()
    for epoch in range(epochs):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            if config["input_noise"] > 0:
                inputs = inputs + config["input_noise"] * torch.randn_like(inputs)
            if config["weight_noise"] > 0:
                clean_weights = [w.detach().clone() for w in weight_matrices]
                with torch.no_grad():
                    for w in weight_matrices:
                        w.add_(config["weight_noise"] * w.abs() * torch.randn_like(w))
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            if config["weight_noise"] > 0:
                with torch.no_grad():
                    for w, clean in zip(weight_matrices, clean_weights):
                        w.copy_(clean)
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)

        if history["val_loss"][-1] < best_loss:
            best_loss, best_epoch, wait = history["val_loss"][-1], epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if config["patience"] is not None and wait >= config["patience"]:
                break
    train_time = time.time() - start
    final = len(history["val_acc"]) - 1
    if config["patience"] is not None:
        model.load_state_dict(best_state)
        final = best_epoch - 1
    model.eval()
    result = {"seed": seed, "train_acc": history["train_acc"][final], "val_acc": history["val_acc"][final],
              "val_loss": history["val_loss"][final], "gap": history["train_acc"][final] - history["val_acc"][final],
              "train_time_s": round(train_time, 1)}
    return model, history, result

BASELINE = {"augment": False, "dropout": 0.0, "label_smoothing": 0.0, "weight_decay": 0.0,
            "input_noise": 0.0, "weight_noise": 0.0, "patience": None}
# The techniques that beat the baseline on validation in Parts 1–10, at their best settings.
# These defaults are placeholders: replace them with your own results (at least four techniques switched on).
COMBINED = {"augment": True, "dropout": 0.25, "label_smoothing": 0.1, "weight_decay": 0.05,
            "input_noise": 0.0, "weight_noise": 0.0, "patience": None}


def describe(config):
    on = [k for k, v in config.items() if v not in (False, 0, 0.0, None)]
    return ", ".join(f"{k}={config[k]}" for k in on) or "baseline"


baseline_models, combined_models, rows = [], [], []
for name, config, keep in [("baseline", BASELINE, baseline_models), ("full combined model", COMBINED, combined_models)]:
    results = []
    for seed in SEEDS:
        model, history, result = train_config(config, seed)
        keep.append(model)
        results.append(result)
    res = pd.DataFrame(results)
    rows.append({"configuration": name, "settings": describe(config),
                 "val_acc": res.val_acc.mean(), "val_acc_sd": res.val_acc.std(), "val_loss": res.val_loss.mean(),
                 "gap": res.gap.mean()})

# Leave-one-out ablation: switch off one technique at a time
switched_on = [k for k, v in COMBINED.items() if v not in (False, 0, 0.0, None)]
for technique in switched_on:
    config = dict(COMBINED)
    config[technique] = BASELINE[technique]
    results = pd.DataFrame([train_config(config, seed)[2] for seed in SEEDS])
    rows.append({"configuration": f"− {technique}", "settings": describe(config),
                 "val_acc": results.val_acc.mean(), "val_acc_sd": results.val_acc.std(),
                 "val_loss": results.val_loss.mean(), "gap": results.gap.mean()})

ablation = pd.DataFrame(rows)
full_acc = ablation.loc[ablation.configuration == "full combined model", "val_acc"].item()
ablation["Δ vs full (pts)"] = (100 * (ablation.val_acc - full_acc)).round(2)
display(ablation.round(4))

**Reading the ablation.** A negative Δ means removing the technique *hurt*, so it contributes. A Δ within about one standard deviation of zero means it is redundant given the others. A positive Δ means it *interferes*: the model is better without it.

**Discussion (model answer): interactions.** Techniques tuned one at a time rarely add up, because several regularize through the same channel:

- **Augmentation and input noise** both perturb the inputs; once crops and flips are present, extra pixel noise adds little.
- **Label smoothing and Mixup** both soften the targets; together they make the network underconfident.
- **Heavy augmentation and large dropout** both slow fitting. Within a fixed epoch budget, stacking them can push the model into underfitting, visible as training accuracy barely above validation accuracy.
- **Early stopping and augmentation:** with augmentation, validation loss reaches its minimum much later, so the patience chosen without augmentation can stop training too soon.
- **Weight decay and dropout** both reduce effective capacity; strengths tuned individually are usually too strong in combination.

The fix is to retune a few of the combined strengths together (for example, halve dropout and weight decay) on the validation set, which is still allowed because the test set has not been used.

In [ ]:
# ===================== FINAL TEST EVALUATION: run once, after every choice above is frozen =====================
# Uses `baseline_models` and `combined_models` from the cell above.
NOISE_SEVERITIES = [0.04, 0.08]      # std of Gaussian noise in [0, 1] pixel units, chosen before testing

test_sets = {}
for severity in [0.0] + NOISE_SEVERITIES:
    rng = np.random.default_rng(123)
    images = X_test.astype(np.float32) / 255
    if severity > 0:
        images = np.clip(images + rng.normal(0, severity, images.shape).astype(np.float32), 0, 1)
    x = ((torch.tensor(images) - MEAN) / STD).permute(0, 3, 1, 2).contiguous()
    test_sets["clean" if severity == 0 else f"noise σ={severity}"] = x
Yte = torch.tensor(yf_test)

rows, test_probs = [], {}
for name, models_list in [("baseline", baseline_models), ("combined", combined_models)]:
    for set_name, X in test_sets.items():
        per_seed = []
        for model in models_list:
            model.eval()
            with torch.no_grad():
                probs = torch.cat([F.softmax(model(X[i:i + 1000].to(device)), 1).cpu() for i in range(0, len(X), 1000)])
            if set_name == "clean":
                test_probs.setdefault(name, probs)
            correct = (probs.argmax(1) == Yte).float()
            confidence = probs.max(1).values
            ece = 0.0
            bin_edges = torch.linspace(0, 1, 16)
            for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
                in_bin = (confidence > lo) & (confidence <= hi)
                if in_bin.any():
                    ece += in_bin.float().mean().item() * abs(confidence[in_bin].mean().item() - correct[in_bin].mean().item())
            per_seed.append({"acc": correct.mean().item(),
                             "loss": -torch.log(probs[torch.arange(len(Yte)), Yte].clamp_min(1e-12)).mean().item(),
                             "ECE": ece})
        df = pd.DataFrame(per_seed)
        rows.append({"model": name, "test set": set_name,
                     "test acc (%)": f"{100 * df.acc.mean():.2f} ± {100 * df.acc.std():.2f}",
                     "test loss": f"{df.loss.mean():.3f} ± {df.loss.std():.3f}",
                     "test ECE": f"{df.ECE.mean():.3f} ± {df.ECE.std():.3f}"})
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot([0, 1], [0, 1], ":", color="gray")
for name, probs in test_probs.items():
    confidence, prediction = probs.max(dim=1)
    correct = (prediction == Yte).float()
    xs, ys = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        in_bin = (confidence > lo) & (confidence <= hi)
        if in_bin.any():
            xs.append(confidence[in_bin].mean().item())
            ys.append(correct[in_bin].mean().item())
    ax.plot(xs, ys, "o-", label=name)
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title("Reliability on the test set (first seed)")
ax.legend()
plt.tight_layout()
plt.show()

**Discussion (model answer).** The combined model should beat the baseline on clean test accuracy by a margin several times larger than the seed-to-seed standard deviation, with much lower loss and ECE; test numbers track validation numbers closely because nothing was tuned on the test set. Its reliability curve sits much closer to the diagonal. Under Gaussian noise both models lose accuracy as severity increases, but the combined model typically degrades less, especially if it includes input noise or augmentation, because those train it on perturbed images. If a group's combined model does *not* beat the baseline, the ablation table is the evidence for why (usually interference), and that is an acceptable, gradable result.

## Part 12: Does regularization matter at scale? Scaling and compute

Regularization is most valuable when data is scarce; this part measures how that changes as the training set grows.

- Train the baseline and the combined model (an MLP, no transfer learning) on nested, stratified 10%, 25%, 50% and 100% subsets of the training split. The validation set stays fixed. Early stopping is turned off here so that training budgets are exactly controlled. Set `COMBINED` to the same settings as in Part 11.
- Two budgets: **fixed epochs** (smaller subsets also get fewer optimizer steps) and **fixed steps** (every subset gets as many optimizer steps as the full data). Comparing them separates the effect of more data from the effect of more optimization.
- One seed per point. Report throughput and total training time.

In [ ]:
from torchvision import transforms


class AugmentedDataset(Dataset):
    def __init__(self, X, y, transform):
        self.X, self.y, self.transform = X, y, transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.transform(self.X[i]), self.y[i]


crop_and_flip = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()])


def train_config(config, seed, epochs=EPOCHS, train_positions=None):
    """Trains the MLP with whichever techniques are switched on in `config`.
    Returns the trained model, its history, and the train/validation numbers of the returned weights."""
    X, Y = Xtr, Ytr
    if train_positions is not None:
        X, Y = Xtr[train_positions], Ytr[train_positions]
    torch.manual_seed(seed)
    p = config["dropout"]
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 32 * 3, 512),
        nn.ReLU(),
        nn.Dropout(p),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(p),
        nn.Linear(256, 100),
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=config["weight_decay"])
    criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])
    dataset = AugmentedDataset(X, Y, crop_and_flip) if config["augment"] else TensorDataset(X, Y)
    train_loader = DataLoader(dataset, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(seed))
    weight_matrices = [w for w in model.parameters() if w.dim() > 1]
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_loss, best_epoch, wait = float("inf"), 0, 0
    best_state = copy.deepcopy(model.state_dict())
    start = time.time()
    for epoch in range(epochs):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            if config["input_noise"] > 0:
                inputs = inputs + config["input_noise"] * torch.randn_like(inputs)
            if config["weight_noise"] > 0:
                clean_weights = [w.detach().clone() for w in weight_matrices]
                with torch.no_grad():
                    for w in weight_matrices:
                        w.add_(config["weight_noise"] * w.abs() * torch.randn_like(w))
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            if config["weight_noise"] > 0:
                with torch.no_grad():
                    for w, clean in zip(weight_matrices, clean_weights):
                        w.copy_(clean)
            optimizer.step()
            loss_sum += loss.item() * len(targets)
            correct += (outputs.argmax(1) == targets).sum().item()
            count += len(targets)
        history["train_loss"].append(loss_sum / count)
        history["train_acc"].append(correct / count)

        model.eval()
        loss_sum, correct, count = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss_sum += F.cross_entropy(outputs, targets).item() * len(targets)
                correct += (outputs.argmax(1) == targets).sum().item()
                count += len(targets)
        history["val_loss"].append(loss_sum / count)
        history["val_acc"].append(correct / count)

        if history["val_loss"][-1] < best_loss:
            best_loss, best_epoch, wait = history["val_loss"][-1], epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if config["patience"] is not None and wait >= config["patience"]:
                break
    train_time = time.time() - start
    final = len(history["val_acc"]) - 1
    if config["patience"] is not None:
        model.load_state_dict(best_state)
        final = best_epoch - 1
    model.eval()
    result = {"seed": seed, "train_acc": history["train_acc"][final], "val_acc": history["val_acc"][final],
              "val_loss": history["val_loss"][final], "gap": history["train_acc"][final] - history["val_acc"][final],
              "train_time_s": round(train_time, 1)}
    return model, history, result

BASELINE = {"augment": False, "dropout": 0.0, "label_smoothing": 0.0, "weight_decay": 0.0,
            "input_noise": 0.0, "weight_noise": 0.0, "patience": None}
# The techniques that beat the baseline on validation in Parts 1–10, at their best settings.
# These defaults are placeholders: replace them with your own results (at least four techniques switched on).
COMBINED = {"augment": True, "dropout": 0.25, "label_smoothing": 0.1, "weight_decay": 0.05,
            "input_noise": 0.0, "weight_noise": 0.0, "patience": None}
COMBINED["patience"] = None          # no early stopping here, so budgets are exactly controlled

# Nested stratified subsets: the same per-class shuffle, so 10% ⊂ 25% ⊂ 50% ⊂ 100%
rng = np.random.default_rng(1)
labels = Ytr.numpy()
class_orders = [rng.permutation(np.flatnonzero(labels == c)) for c in range(100)]

rows = []
for frac in [0.1, 0.25, 0.5, 1.0]:
    positions = np.sort(np.concatenate([order[: max(1, round(frac * len(order)))] for order in class_orders]))
    for name, config in [("baseline", BASELINE), ("combined", COMBINED)]:
        for budget, epochs in [("fixed epochs", EPOCHS), ("fixed steps", round(EPOCHS / frac))]:
            if frac == 1.0 and budget == "fixed steps":
                continue                              # identical to fixed epochs at 100%
            model, history, result = train_config(config, seed=0, epochs=epochs, train_positions=positions)
            row = {"model": name, "budget": budget, "n_train": len(positions), "epochs": epochs,
                   "steps": epochs * math.ceil(len(positions) / 128), **result,
                   "images/s": round(epochs * len(positions) / result["train_time_s"])}
            rows.append(row)
            if frac == 1.0:
                rows.append({**row, "budget": "fixed steps"})
scale_df = pd.DataFrame(rows)
display(scale_df.drop(columns="seed").round(4))

fig, axs = plt.subplots(1, 3, figsize=(16, 4.2))
for (name, budget), g in scale_df.groupby(["model", "budget"]):
    style = "o-" if budget == "fixed steps" else "o--"
    axs[0].plot(g.n_train, 100 * g.val_acc, style, label=f"{name}, {budget}")
    axs[1].plot(g.n_train, 100 * g.gap, style, label=f"{name}, {budget}")
    axs[2].plot(g.train_time_s, 100 * g.val_acc, style, label=f"{name}, {budget}")
axs[0].set_ylabel("validation accuracy (%)"); axs[1].set_ylabel("train − validation accuracy (pts)")
axs[2].set_ylabel("validation accuracy (%)"); axs[2].set_xlabel("training time (s)")
for ax in axs[:2]:
    ax.set_xscale("log"); ax.set_xlabel("training images (log scale)")
for ax in axs:
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

# Extrapolation: fit baseline accuracy ≈ a + b·ln(n) under fixed steps, and solve for the n at which the baseline
# would match the combined model's accuracy at the smallest subset.
g = scale_df[(scale_df.model == "baseline") & (scale_df.budget == "fixed steps")].sort_values("n_train")
b, a = np.polyfit(np.log(g.n_train), g.val_acc, 1)
smallest = scale_df[(scale_df.model == "combined") & (scale_df.budget == "fixed steps")].sort_values("n_train").iloc[0]
n_needed = math.exp((smallest.val_acc - a) / b) if b > 0 else float("nan")
if b > 0 and n_needed > smallest.n_train:
    print(f"combined model with {smallest.n_train} images: {smallest.val_acc:.2%}. The fit says the baseline would need "
          f"about {n_needed:,.0f} images to match it ({n_needed / smallest.n_train:.1f}× more data).")
else:
    print("At the smallest subset the combined model does not beat the baseline, so there is nothing to extrapolate.")

device_name = torch.cuda.get_device_name(0) if device.type == "cuda" else str(device)
print(f"\nhardware: {device_name} | batch size 128 | total training time in this part: "
      f"{scale_df.drop_duplicates(['model', 'n_train', 'epochs']).train_time_s.sum() / 3600:.2f} hours")

In [ ]:
# Data loading throughput: does loading with parallel workers help? (loading and moving to the device only, no model)
from torchvision import transforms


class AugmentedDataset(Dataset):
    def __init__(self, X, y, transform):
        self.X, self.y, self.transform = X, y, transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.transform(self.X[i]), self.y[i]


crop_and_flip = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()])
for name, dataset in [("no augmentation", TensorDataset(Xtr, Ytr)), ("crop + flip", AugmentedDataset(Xtr, Ytr, crop_and_flip))]:
    for workers in [0, 2]:
        try:
            loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=workers,
                                pin_memory=device.type == "cuda")
            start = time.time()
            for inputs, targets in loader:
                inputs = inputs.to(device)
            print(f"{name}, num_workers={workers}: {len(dataset) / (time.time() - start):,.0f} images/s")
        except Exception as e:      # some notebook environments (for example macOS) cannot start worker processes
            print(f"{name}, num_workers={workers}: not available here ({type(e).__name__})")

**Discussion (model answer).**

*Does the regularization gap shrink with data?* Yes. Under fixed steps, both models improve roughly linearly in log(training images), and the baseline improves faster, so the distance between the curves narrows as data grows. The baseline's train − validation gap also shrinks, because 40,000 images are harder to memorize than 4,000. Regularization is a substitute for data: the combined model at a small subset matches what the extrapolated baseline only reaches with several times more images (the cell prints the estimate, which is a rough guide fitted to four points).

*Data vs. steps.* Under fixed epochs, the small subsets also receive far fewer optimizer steps, so their low accuracy mixes "less data" with "less training". The fixed-steps curves remove that confound. For the baseline, extra steps on a small subset mostly increase overfitting (high training accuracy, unchanged validation accuracy), which shows that the limit at small n is data, not optimization.

*What to prioritize at 100× the data.*
1. **Transfer learning / pretraining** stays the most valuable, because it brings in information from outside the dataset.
2. **Data augmentation** stays useful. It encodes invariances (shift, mirror) that more data covers only gradually, and large vision models are still trained with heavy augmentation.
3. **Weight decay** is free and remains standard.
4. **Early stopping** becomes a compute-saving tool more than a regularizer.
5. **Dropout and label smoothing** matter less as overfitting fades, and **ensembling** becomes the most expensive option, since every member must be trained on the full data.

*Throughput.* Per-image augmentation in a `Dataset` costs CPU time, and loader worker processes help when that becomes the bottleneck; for data that does not fit in memory, parallel loading is essential.

*Limitations.* CIFAR-100 fits in memory and trains in minutes, so it cannot show the problems that dominate at real scale: I/O and storage throughput, distributed training, data deduplication, label noise and long-tailed classes. The trends here (regularization matters less with more data; augmentation and pretraining keep mattering) are consistent with large-scale results, but the magnitudes do not transfer.

## Results table

Fill this in from the "repeat with 3 seeds" cells of each part, Part 11 and the transfer and ensemble results:

| Experiment | Main setting | Validation accuracy | Test accuracy* | Test NLL* | Calibration error* | Training time | Inference cost |
|---|---|---:|---:|---:|---:|---:|---:|
| Baseline | — | | | | | | |
| Each individual method | | | | | | | |
| Combined model | | | | | | | |
| Ablations | | | | | | | |

\*Populate test columns only for experiments included in your frozen final evaluation plan. Report repeated-run results as mean ± standard deviation.

## Deliverables

### 1. Reproducible implementation

Submit a repository or archive containing:

- Dataset loading and saved split indices (`splits.npz`).
- Training, evaluation, and ensemble code.
- Configuration files for every reported experiment.
- Random seeds, dependency versions, and execution instructions.
- Experiment logs and final model checkpoints, or an accessible checkpoint link.

### 2. Technical report

Submit a report of **6–8 pages**, excluding references and appendices, covering:

- Experimental design and compute budget.
- Baseline behavior and evidence of overfitting or underfitting.
- Individual-method results.
- Combined-model results and ablations.
- Calibration, robustness, and scaling results.
- At least two unsuccessful experiments and what they reveal.
- Limitations and your recommended model under a realistic compute budget.

### 3. Short presentation

Prepare a **five-minute presentation** explaining your most important finding, supported by one results table and two plots.

## Grading

| Component | Weight |
|---|---:|
| Correct implementation of the ten methods | 30% |
| Controlled design and reproducibility | 20% |
| Evaluation, repeated runs, and ablations | 20% |
| Interpretation of successful and unsuccessful results | 15% |
| Scaling and computational analysis | 10% |
| Report and presentation clarity | 5% |

**Grading rewards sound experiments and defensible conclusions, not the highest accuracy.** Data leakage, test-set tuning, and selective reporting undermine the central purpose of the assignment.